# Stage 1: Data Profiling & Quality Exploration
This notebook inspects, profiles, and validates the three raw datasets:
1. `data/product_information.csv` (Catalogue)
2. `data/discount.csv` (Product-City Discount Rates)
3. `data/products_sold.csv` (Transactional Sales)

The goal is to verify row counts, schemas, candidate keys, nulls, value ranges, duplicate vs. conflicting records, and referential integrity before designing the data warehouse schema and ETL pipeline. Analytical answers belong in the final challenge notebook, not here.


In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"
PROD_CSV = DATA_DIR / "product_information.csv"
DISC_CSV = DATA_DIR / "discount.csv"
SOLD_CSV = DATA_DIR / "products_sold.csv"

EXPECTED_COLUMNS = {
    PROD_CSV: [
        'Product ID', 'Product Position', 'Promotion', 'Product Category', 'Seasonal',
        'brand', 'url', 'name', 'description', 'price', 'currency', 'terms',
        'section', 'season', 'material', 'origin',
    ],
    DISC_CSV: ['Product ID', 'city', 'discount'],
    SOLD_CSV: ['Product ID', 'pieces_sold', 'city', 'time'],
}

def column_profile(df):
    """Compact completeness, type, and cardinality profile."""
    rows = []
    for column in df.columns:
        series = df[column]
        is_text = pd.api.types.is_string_dtype(series)
        non_null = series.dropna()
        rows.append({
            'column': column,
            'dtype': str(series.dtype),
            'nulls': int(series.isna().sum()),
            'distinct': int(series.nunique(dropna=True)),
            'blank_strings': int(non_null.str.strip().eq('').sum()) if is_text else 0,
            'padded_strings': int(non_null.ne(non_null.str.strip()).sum()) if is_text else 0,
        })
    return pd.DataFrame(rows)

print(f"Base Directory: {BASE_DIR}")


Base Directory: /Users/amroelsaadany/Documents/GitHub/retail-sales-warehouse


## 1. Raw Line Count and CSV Parsing Verification

In [2]:
parse_results = []
for path, expected_header in EXPECTED_COLUMNS.items():
    with open(path, "r", encoding="utf-8", newline="") as file:
        reader = csv.reader(file, strict=True)
        header = next(reader)
        row_count = 0
        malformed_rows = 0
        for row in reader:
            row_count += 1
            malformed_rows += len(row) != len(expected_header)
    parse_results.append({
        'file': path.name,
        'rows': row_count,
        'columns': len(header),
        'duplicate_headers': len(header) - len(set(header)),
        'missing_headers': sorted(set(expected_header) - set(header)),
        'extra_headers': sorted(set(header) - set(expected_header)),
        'exact_header_set': (
            len(header) == len(set(header))
            and set(header) == set(expected_header)
        ),
        'malformed_rows': malformed_rows,
    })

parse_summary = pd.DataFrame(parse_results)
display(parse_summary)
assert (parse_summary['malformed_rows'] == 0).all()
assert parse_summary['exact_header_set'].all()


,file,rows,columns,duplicate_headers,missing_headers,extra_headers,exact_header_set,malformed_rows
0,product_information.csv,20252,16,0,[],[],True,0
1,discount.csv,185962,3,0,[],[],True,0
2,products_sold.csv,1010492,4,0,[],[],True,0


## 2. Product Information Profiling (`product_information.csv`)

In [3]:
df_prod = pd.read_csv(PROD_CSV)
display(column_profile(df_prod))
print(f"Rows: {len(df_prod):,}")
print(f"Unique Product IDs: {df_prod['Product ID'].nunique():,}")
print(f"Exact duplicate rows: {df_prod.duplicated().sum():,}")
print(f"Duplicate Product IDs: {df_prod.duplicated(subset=['Product ID']).sum():,}")
assert pd.api.types.is_integer_dtype(df_prod['Product ID'])
assert pd.api.types.is_numeric_dtype(df_prod['price'])
assert df_prod['Product ID'].is_unique


,column,dtype,nulls,distinct,blank_strings,padded_strings
0,Product ID,int64,0,20252,0,0
1,Product Position,str,0,3,0,0
2,Promotion,str,0,2,0,0
3,Product Category,str,0,1,0,0
4,Seasonal,str,0,2,0,0
5,brand,str,0,1,0,0
6,url,str,0,228,0,0
7,name,str,1,17215,0,0
8,description,str,2,221,0,0
9,price,float64,0,330,0,0


Rows: 20,252
Unique Product IDs: 20,252
Exact duplicate rows: 0
Duplicate Product IDs: 0


In [4]:
missing_products = df_prod[df_prod.isnull().any(axis=1)]
display(missing_products[['Product ID', 'name', 'description']])

print("Price statistics:")
display(df_prod['price'].describe().to_frame().T)
assert df_prod['price'].gt(0).all()
assert np.isfinite(df_prod['price']).all()


,Product ID,name,description
60,151925,VINTAGE EFFECT LEATHER BOMBER JACKET,NaN
72,173576,NaN,NaN


Price statistics:


,count,mean,std,min,25%,50%,75%,max
price,20252.0,41.949061,23.38096,12.0,23.95,35.95,53.95,134.99


In [5]:
product_dimensions = [
    'currency', 'brand', 'section', 'season', 'terms', 'material', 'origin',
    'Product Position', 'Promotion', 'Product Category', 'Seasonal',
]
for column in product_dimensions:
    print(f"{column}: {df_prod[column].value_counts(dropna=False).to_dict()}")


currency: {'USD': 20252}
brand: {'Zara': 20252}
section: {'WOMAN': 13254, 'MAN': 6998}
season: {'Autumn': 7665, 'Winter': 5144, 'Spring': 4537, 'Summer': 2906}
terms: {'jackets': 11232, 'sweaters': 3257, 't-shirts': 2646, 'shoes': 2458, 'jeans': 659}
material: {'Cotton': 3851, 'Wool': 3805, 'Wool Blend': 3373, 'Polyester': 2775, 'Linen': 2573, 'Denim': 1027, 'Viscose': 990, 'Acrylic': 881, 'Linen Blend': 807, 'Satin': 132, 'Silk': 38}
origin: {'China': 4026, 'Bangladesh': 3617, 'Turkey': 2475, 'India': 2033, 'Morocco': 1653, 'Portugal': 1420, 'Spain': 1248, 'Vietnam': 1220, 'Cambodia': 981, 'Brazil': 795, 'Pakistan': 605, 'Argentina': 179}
Product Position: {'Aisle': 7810, 'End-cap': 6791, 'Front of Store': 5651}
Promotion: {'No': 11812, 'Yes': 8440}
Product Category: {'clothing': 20252}
Seasonal: {'No': 10136, 'Yes': 10116}


## 3. Discount Profiling (`discount.csv`)

In [6]:
df_disc = pd.read_csv(DISC_CSV)
discount_key = ['Product ID', 'city']
display(column_profile(df_disc))

exact_discount_duplicates = int(df_disc.duplicated().sum())
discount_rate_counts = df_disc.groupby(discount_key, dropna=False)['discount'].nunique(dropna=False)
conflicting_pairs = discount_rate_counts[discount_rate_counts > 1]

print(f"Rows: {len(df_disc):,}")
print(f"Exact duplicate rows: {exact_discount_duplicates:,}")
print(f"Unique product-city pairs: {len(discount_rate_counts):,}")
print(f"Product-city pairs with conflicting rates: {len(conflicting_pairs):,}")
print(f"Discount range: {df_disc['discount'].min():.2f} to {df_disc['discount'].max():.2f}")

assert pd.api.types.is_integer_dtype(df_disc['Product ID'])
assert pd.api.types.is_numeric_dtype(df_disc['discount'])
assert df_disc['discount'].between(0, 1).all()
assert np.isfinite(df_disc['discount']).all()
assert conflicting_pairs.empty

# Remove only byte-for-byte equivalent business records. A conflict would fail above.
df_disc_clean = df_disc.drop_duplicates()
assert not df_disc_clean.duplicated(discount_key).any()


,column,dtype,nulls,distinct,blank_strings,padded_strings
0,Product ID,int64,0,20252,0,0
1,city,str,0,10,0,0
2,discount,float64,0,21,0,0


Rows: 185,962
Exact duplicate rows: 1,841
Unique product-city pairs: 184,121
Product-city pairs with conflicting rates: 0
Discount range: 0.00 to 0.20


In [7]:
display(
    df_disc_clean.groupby('city')['discount']
    .agg(product_count='count', min_discount='min', mean_discount='mean', max_discount='max')
    .sort_index()
)


,product_count,min_discount,mean_discount,max_discount
city,,,,
Chicago,18468,0.0,0.100535,0.2
Dallas,18419,0.0,0.100378,0.2
Houston,18428,0.0,0.100312,0.2
Los Angeles,18394,0.0,0.099295,0.2
New York,18425,0.0,0.099316,0.2
Philadelphia,18379,0.0,0.099848,0.2
Phoenix,18477,0.0,0.100630,0.2
San Antonio,18395,0.0,0.099787,0.2
San Diego,18360,0.0,0.099987,0.2


## 4. Products Sold Profiling (`products_sold.csv`)

In [8]:
df_sold = pd.read_csv(SOLD_CSV)
display(column_profile(df_sold))

candidate_sales_key = ['Product ID', 'city', 'time']
print(f"Rows: {len(df_sold):,}")
print(f"Exact duplicate rows: {df_sold.duplicated().sum():,}")
print(f"Duplicate candidate keys: {df_sold.duplicated(candidate_sales_key).sum():,}")

assert pd.api.types.is_integer_dtype(df_sold['Product ID'])
assert pd.api.types.is_integer_dtype(df_sold['pieces_sold'])
assert pd.api.types.is_integer_dtype(df_sold['time'])
assert df_sold['pieces_sold'].gt(0).all()
assert df_sold['pieces_sold'].mod(1).eq(0).all()


,column,dtype,nulls,distinct,blank_strings,padded_strings
0,Product ID,int64,0,20252,0,0
1,pieces_sold,int64,0,99,0,0
2,city,str,0,10,0,0
3,time,int64,0,993091,0,0


Rows: 1,010,492
Exact duplicate rows: 0
Duplicate candidate keys: 0


In [9]:
print("Pieces Sold Summary:")
print(df_sold['pieces_sold'].describe())

df_sold['datetime_utc'] = pd.to_datetime(df_sold['time'], unit='s', utc=True, errors='coerce')
assert df_sold['datetime_utc'].notna().all()
print(f"Timestamp range (UTC): {df_sold['datetime_utc'].min()} to {df_sold['datetime_utc'].max()}")
covered_days = df_sold['datetime_utc'].dt.floor('D').unique()
expected_days = pd.date_range(
    df_sold['datetime_utc'].min().floor('D'),
    df_sold['datetime_utc'].max().floor('D'),
    freq='D',
)
print(f"Missing calendar days inside observed range: {len(expected_days.difference(covered_days))}")
print("Sales transactions per month:")
print(df_sold['datetime_utc'].dt.strftime('%Y-%m').value_counts().sort_index())

Pieces Sold Summary:
count    1.010492e+06
mean     6.623444e+01
std      2.347536e+01
min      1.000000e+00
25%      5.000000e+01
50%      7.000000e+01
75%      8.600000e+01
max      9.900000e+01
Name: pieces_sold, dtype: float64
Timestamp range (UTC): 2025-01-01 00:00:04+00:00 to 2025-11-29 23:59:10+00:00
Missing calendar days inside observed range: 0
Sales transactions per month:


datetime_utc
2025-01    94383
2025-02    85066
2025-03    93975
2025-04    90341
2025-05    94562
2025-06    91079
2025-07    93570
2025-08    94003
2025-09    91499
2025-10    93771
2025-11    88243
Name: count, dtype: int64


## 5. Referential Integrity & Coverage Analysis

In [10]:
prod_set = set(df_prod['Product ID'])
sold_prod_set = set(df_sold['Product ID'])
disc_prod_set = set(df_disc['Product ID'])

print(f"Products in Sales not in Catalogue: {len(sold_prod_set - prod_set)}")
print(f"Products in Discounts not in Catalogue: {len(disc_prod_set - prod_set)}")
print(f"Catalogue products with NO sales: {len(prod_set - sold_prod_set)}")

print(f"Sales cities equal discount cities: {set(df_sold['city']) == set(df_disc_clean['city'])}")

raw_discount_join_rows = len(df_sold.merge(df_disc, on=discount_key, how='left'))
sales_with_discount = df_sold.merge(
    df_disc_clean,
    on=discount_key,
    how='left',
    validate='many_to_one',
    indicator=True,
)
sales_with_product = df_sold.merge(
    df_prod[['Product ID']],
    on='Product ID',
    how='left',
    validate='many_to_one',
    indicator=True,
)

missing_discount_mask = sales_with_discount['_merge'].eq('left_only')
missing_discounts = int(missing_discount_mask.sum())
missing_discount_pairs = int(
    sales_with_discount.loc[missing_discount_mask, discount_key].drop_duplicates().shape[0]
)
missing_products = int(sales_with_product['_merge'].eq('left_only').sum())
print(f"Raw discount join rows: {raw_discount_join_rows:,} ({raw_discount_join_rows - len(df_sold):,} too many)")
print(f"Deduplicated discount join rows: {len(sales_with_discount):,}")
print(f"Sales rows without a source discount mapping: {missing_discounts:,}")
print(f"Distinct missing product-city discount pairs: {missing_discount_pairs:,}")
print(f"Sales rows without a catalogue product: {missing_products:,}")
print(f"Units before join: {df_sold['pieces_sold'].sum():,}")
print(f"Units after join:  {sales_with_discount['pieces_sold'].sum():,}")

assert len(sales_with_discount) == len(df_sold)
assert sales_with_discount['pieces_sold'].sum() == df_sold['pieces_sold'].sum()
assert missing_discounts == 0
assert missing_products == 0


Products in Sales not in Catalogue: 0
Products in Discounts not in Catalogue: 0
Catalogue products with NO sales: 0
Sales cities equal discount cities: True


Raw discount join rows: 1,020,777 (10,285 too many)
Deduplicated discount join rows: 1,010,492
Sales rows without a source discount mapping: 0
Distinct missing product-city discount pairs: 0
Sales rows without a catalogue product: 0
Units before join: 66,929,370
Units after join:  66,929,370


## 6. Task 2 Profiling Implication

In [11]:
sales_by_product = df_sold.groupby('Product ID')['pieces_sold'].sum()
min_units = int(sales_by_product.min())
max_units = int(sales_by_product.max())
print(f"Best-selling total: {max_units:,} units; tied products: {sales_by_product.eq(max_units).sum():,}")
print(f"Worst-selling total: {min_units:,} unit; tied products: {sales_by_product.eq(min_units).sum():,}")
print("Task 2 must retain ties rather than selecting one arbitrary product.")


Best-selling total: 9,801 units; tied products: 202
Worst-selling total: 1 unit; tied products: 201
Task 2 must retain ties rather than selecting one arbitrary product.


## 7. Findings and Proposed Cleaning Rules

### Findings

- The catalogue contains one row per `Product ID`; IDs are complete and unique. One name and two descriptions are null. These descriptive fields are optional for the analysis.
- Catalogue prices are positive and all currencies are USD. The source supplies a current product price, not a historical transaction price. Product attributes and prices form a static snapshot because no effective dates exist.
- Discounts are fractional rates from 0.00 to 0.20. The 1,841 extra rows are exact duplicates. There are no conflicting rates for a product-city pair.
- Sales contain positive integer quantities and Unix-second timestamps interpreted as UTC. The observed `(Product ID, city, time)` combination is unique, but no transaction identifier or source guarantee justifies enforcing it as a business key.
- Every sales and discount product exists in the catalogue. Every sale currently has a matching source product-city discount, although the data contract allows a mapping to be absent. Every catalogue product appears in sales in this extract.
- Joining sales directly to raw discounts would multiply rows. Exact discount deduplication restores a many-to-one join and preserves both sales rows and total units.
- The extract ends on 29 November 2025, so November is incomplete and December is absent.

### Proposed rules for the pipeline

1. Parse with a CSV-aware reader. Require the exact header-name set, allow any column order, and reject missing, extra, or duplicate names.
2. Preserve null product names and descriptions; do not invent replacements.
3. Reject invalid IDs, non-positive prices or quantities, discount rates outside 0–1, invalid timestamps, and orphan product references.
4. Remove only exact discount duplicates and retain all supplied mappings, including mappings unused by current sales. If a product-city pair has multiple rates, fail validation and require an explicit business decision; never choose the maximum silently.
5. If a sale has no source product-city discount mapping, accept it and count the affected sales and distinct missing pairs. Keep the mapping absent for lineage; analytical SQL supplies zero with `COALESCE`.
6. Preserve every accepted sales row and its full UTC timestamp. Do not impose uniqueness without a business identifier.
7. Treat catalogue price as the assumed unit sale price for the challenge and label monetary results accordingly.

These rules are the Stage 1 proposal to carry into schema and pipeline design.
